# Resource Allocation Optimization Agent

This agent takes the power of Azure OpenAI's reasoning models to solve complex optimization challenges, common to many enterprise scenarios. One such common optimization challenge is resource allocation to tasks. In case you have a process with well defined task list, with inputs and outputs, assigning a specialist with approriate skills to resolve that task is often a human endevour. This agent will aim to take that human task of allocating the resources, and use the power of reasoning models to complete the task.



In [1]:
import os
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from azure.ai.projects.models import CodeInterpreterTool
from azure.ai.projects.models import FilePurpose
import time
from dotenv import load_dotenv
from datetime import datetime
from IPython.display import Markdown, display

load_dotenv()

True

In [2]:
# Helper function
def get_conversation_md(conversation):
    """
    Function to return a conversation in Markdown (MD) format as a string.
    """
    messages = conversation.get("data", [])
    if not messages:
        return "No messages found in the conversation."

    # Initialize a list to hold the Markdown lines
    md_lines = []
    md_lines.append("# Conversation")
    md_lines.append("___")  # Markdown horizontal line

    # Iterate through the messages
    # Reversing to maintain chronological order
    for message in reversed(messages):
        role = message.get("role", "unknown").capitalize()
        timestamp = message.get("created_at")
        content = message.get("content", [])

        # Convert timestamp to a readable format
        if timestamp:
            timestamp = datetime.fromtimestamp(
                timestamp).astimezone().strftime('%Y-%m-%d %H:%M:%S %Z')
        else:
            timestamp = "Unknown time"

        # Extract the text content
        message_text = ""
        for item in content:
            if item.get("type") == "text":
                message_text += item["text"].get("value", "")

        # Append the message in Markdown format
        md_lines.append(f"### **{role}** ({timestamp})")
        md_lines.append(f"{message_text}")
        md_lines.append("___")  # Markdown horizontal line

    # Join the lines with newlines to form the complete Markdown string
    return "\n".join(md_lines)

In [3]:
# Create a client to interact with the Azure AI service
project_client = AIProjectClient.from_connection_string(
    credential=DefaultAzureCredential(), conn_str=os.environ["PROJECT_CONNECTION_STRING"]
)

In [4]:
# Upload a file to the project
resource_list = project_client.agents.upload_file_and_poll(
    file_path="./data/sample_resource_list.txt", purpose=FilePurpose.AGENTS
)
print(f"Uploaded file resource_list, file ID: {resource_list.id}")

task_list = project_client.agents.upload_file_and_poll(
    file_path="./data/sample_task_list.txt", purpose=FilePurpose.AGENTS
)
print(f"Uploaded file task_list, file ID: {task_list.id}")

# Create a code interpreter tool
code_interpretter_tool = CodeInterpreterTool(
    file_ids=[resource_list.id, task_list.id],
)

Uploaded file resource_list, file ID: assistant-DbcKV4LAmzBQW9znG1eSRg
Uploaded file task_list, file ID: assistant-Jr6JGwaeEXt9CReAH8gLJN


In [ ]:
INSTRUCTIONS = r"""
<question>
Today's date is 25-Feb-25.
 
As an resource manager at Contoso Mining company, your tasks are:

Identify the mechanical parts(s) that should be prioritized for maintenance based on the downtime risk, downtime cost and engineer availability.
We also need to ensure we do proactive maintenance of each part every 3 months. For each task prioritized you identify, provide a detailed rationale explaining why it should be prioritized in that order.
If you are unable to allocate an engineer to a task due to resource constraints, call that out in a separate section and recommend a specific maintenance or replacement plan for these part(s), considering operational constraints, safety regulations, projected load changes, and potential risks.
You could choose to allocate overtime work for high critical part maintenance if required, but aim to minimize overtime work. 

<\question>

<instructions> 

Allocate engineers to projects to optimize skill matching and minimize total downtime risk weighted costs.  Use the steps below.
1. Prioritize the tasks based on a factor of downtime risk and downtime cost. 
2. Ensure that each part is maintained at least every 3 months, so check the last maintenance date to validate this.
3. Identify specialist and allocate them based on task priority and availability
4. Optimize for reduction in total downtime risk factor.

Prepare the following tables as summary outputs

1. Resource allocation table
engineer allocated | specialization | part | Ticket ID | start date | end date 

2. Resource Utilization summary table
engineer | specialization | initial utilization | final utilization
initial utilization = (80 - initial availability in hours) / 80 
final utilization = ((80 - initial availability in hours) + hours allocated )/ 80


Note: Your analysis should aim to:
Prevent high risk downtime for more than 10 hrs.
Utilize the engineers with the right skills for the right job.
Ensure compliance with safety regulations and operational constraints.
Understand that each engineer is allowed to work a maximum of 8 hrs per day during regular hours, and 12 hours per day with overtime.
 
 </instructions>

<background>

Contoso is a leading manufacturing company, and you are a resource manager with the job of allocating work items to maintenance engineering team, and aim to maximimze their utilization.

Datasets:
You are provided with the following datasets. The data will be in form of txt files and you will need to use the pipe (|) as the delimiter to read the data.
The first line of the file will be the header, and the rest of the lines will be the data.
Resource list:  List of Engineers, with their Name, Specialization and hours of availability per workday for the next 2 weeks
Task List: List of maintenance tasks that need to be completed with Part name, specialization required, ticket ID  (within our ERP system), downtime risk and cost, and estimated hours of time required to fix the part.

</background>

<output>

Prepare the following tables as summary outputs.
Ensure that we have tasks assigned to each engineer. If no new tasks assigned to an engineer, call that out as "NO-TASK".

### Output 1: Resource allocation table
engineer allocated | specialization | part | Ticket ID | start date | end date 


### Output 2: Resource Utilization summary table
Recalculate the utilization of each engineer before and after new task allocation.

engineer | specialization | initial utilization | final utilization

Notes:
initial utilization = (80 - initial availability in hours) / 80 
final utilization = ((80 - initial availability in hours) + hours allocated )/ 80


### Output 3: Unassigned tasks
part | specialist required | ticket id | downtime_risk  | dowtime_cost | potential risk mitigation plan

Notes:
for the risk mitigation plan, suggest possible mitigation actions including overtime assignments or new part purchase.

<\output>
"""

## Create the agent

In [6]:
# Create an agent 
resource_manager = project_client.agents.create_agent(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    name="resource_manager",
    description="An agent that manages resources for maintenance tasks",
    instructions=INSTRUCTIONS,
    tools=code_interpretter_tool.definitions,
    tool_resources=code_interpretter_tool.resources,
    # Parameters
    temperature=1,
    top_p=0.95,
)

print(f"Created agent, agent ID: {resource_manager.id}")

Created agent, agent ID: asst_tXeMUW3BNoAkMvpnlarvSMpm


## Run the Agent

In [7]:
MESSAGE = "Can you help me allocate engineers to the tasks based on the instructions provided?"

In [8]:
# Create the thread
thread = project_client.agents.create_thread()
print(f"Created thread, ID: {thread.id}")

Created thread, ID: thread_EnR8Viawgfohvf0JkdMvaw8M


In [9]:
# Create message to thread
message = project_client.agents.create_message(
    thread_id=thread.id, 
    role="user", 
    content=MESSAGE,
)
print(f"Created message, ID: {message.id}")

Created message, ID: msg_4qQNezm9fXaUJAuTEdlDXdkJ


In [10]:
# Run the agent
# Create and process assistant run in thread with tools
run = project_client.agents.create_run(
    thread_id=thread.id, assistant_id=resource_manager.id)
print(f"Created run, ID: {run.id}")

while run.status in ["queued", "in_progress", "requires_action"]:
    time.sleep(5)
    run = project_client.agents.get_run(thread_id=thread.id, run_id=run.id)
    print(f"Run status: {run.status}")

print(f"Run Completed with status: {run.status}")

messages = project_client.agents.list_messages(thread_id=thread.id)
print(f"Messages in thread: {len(messages)}")

Created run, ID: run_NDWtqmfDYOnoRCJxA1G34iQT
Run status: RunStatus.IN_PROGRESS
Run status: RunStatus.IN_PROGRESS
Run status: RunStatus.IN_PROGRESS
Run status: RunStatus.IN_PROGRESS
Run status: RunStatus.IN_PROGRESS
Run status: RunStatus.IN_PROGRESS
Run status: RunStatus.FAILED
Run Completed with status: RunStatus.FAILED
Messages in thread: 5


In [11]:
display(Markdown(get_conversation_md(messages)))

# Conversation
___
### **User** (2025-03-19 11:30:32 +08)
Can you help me allocate engineers to the tasks based on the instructions provided?
___
### **Assistant** (2025-03-19 11:31:01 +08)
It appears that the data might not have been parsed correctly or might have been merged into a single string. Let's attempt to read the data again with the correct delimiter.

We should try to read the CSV files with the appropriate delimiter and re-examine the data.
___

In [14]:
import json
messages.as_dict()

{'object': 'list',
 'data': [{'id': 'msg_QK5cCOzlr3JyeZ9lXiV0nyvT',
   'object': 'thread.message',
   'created_at': 1742355061,
   'assistant_id': 'asst_tXeMUW3BNoAkMvpnlarvSMpm',
   'thread_id': 'thread_EnR8Viawgfohvf0JkdMvaw8M',
   'run_id': 'run_NDWtqmfDYOnoRCJxA1G34iQT',
   'role': 'assistant',
   'content': [{'type': 'text',
     'text': {'value': "It appears that the data might not have been parsed correctly or might have been merged into a single string. Let's attempt to read the data again with the correct delimiter.\n\nWe should try to read the CSV files with the appropriate delimiter and re-examine the data.",
      'annotations': []}}],
   'attachments': [],
   'metadata': {}},
  {'id': 'msg_4qQNezm9fXaUJAuTEdlDXdkJ',
   'object': 'thread.message',
   'created_at': 1742355032,
   'assistant_id': None,
   'thread_id': 'thread_EnR8Viawgfohvf0JkdMvaw8M',
   'run_id': None,
   'role': 'user',
   'content': [{'type': 'text',
     'text': {'value': 'Can you help me allocate engine